In [ ]:
"""
Historical data collection via Arctic Shift with AND retrieval logic
=======================================================================
For each candidate subreddit (from the candidate discovery list), all
historical submissions within the observation period are retrieved and
filtered using an AND logic:

    (Modality-Term_1 OR Modality-Term_2 OR ...)
    AND
    (Crime-Term_1 OR Crime-Term_2 OR ...)

A post is only flagged as a "candidate hit" if it contains AT LEAST ONE
term from BOTH blocks (in the title or selftext).

IMPORTANT: A keyword hit only means "potentially relevant" (retrieval level),
NOT automatically "relevant in terms of content". The content validation
takes place in a separate, downstream step (manual/sample-based).

Subreddits without a single candidate hit automatically drop out of the
final analysis - a separate upstream screening is therefore not required.

Search terms: extracted automatically from the Excel file "Subreddit_suche_subreddits_
gesucht.xlsx" (columns "eingabe bei reddit" + "seed term"),
split by Main Category "Deepfake" (-> MODALITY_TERMS) and
"Identity" (-> CRIME_TERMS). 107 modality terms, 319 crime terms,
deduplicated.

Run locally (NOT in Google Colab):
    pip install requests pandas langdetect
    python arctic_shift_retrieval.py
"""

import requests
import pandas as pd
import time
from datetime import datetime, timezone

# =============================================================================
# CONFIGURATION
# =============================================================================

REDDIT_USERNAME = "your_reddit_username"   # <-- CHANGE THIS (logging/documentation only)

# Observation period (study-specific, see methods chapter)
START_DATE = datetime(2017, 1, 1, tzinfo=timezone.utc)
END_DATE = datetime(2026, 8, 30, tzinfo=timezone.utc)   # date of data collection

# Permitted languages (a post is only kept if its detected language is in this set)
ALLOWED_LANGUAGES = {"en", "de"}

# -----------------------------------------------------------------------
# MODALITY TERMS (Block A) - extracted automatically from columns
# "eingabe bei reddit" + "seed term" for Main Category "Deepfake"
# -----------------------------------------------------------------------
MODALITY_TERMS = [
    "face swapping",
    "face swap",
    "image face swap",
    "face swap images",
    "image face swap ai",
    "face body swapping",
    "face/body swap",
    "body swap",
    "Non-existent face generation",
    "face generator of non-existent people",
    "face generation non existent people",
    "Entire face synthesis",
    "syntetic face dataset",
    "Face manipulation GAN",
    "gan manipulation face expression",
    "video face swap",
    "face swap videos",
    "video face swap ai",
    "Lip-sync manipulation",
    "ai lip-sync",
    "ai lip sync",
    "Face synthesis",
    "ai faces",
    "synthetic faces",
    "aisynthezied faces",
    "Face morphing",
    "face morph",
    "face-morphing",
    "facial morph",
    "Full body puppetry",
    "full body ai",
    "fullbody ai motion capture",
    "full body images ai",
    "full-body ultra realistic",
    "ai full body mri",
    "Expression swap",
    "facial expression swap",
    "Face re-enactment",
    "face recreation",
    "face reenactment",
    "face reenactment ai",
    "Voice impersonation",
    "ai voice generation",
    "voice impressions",
    "voice imiation",
    "Voice cloning",
    "cole voice",
    "voice creation",
    "ai voice cloning",
    "Voice swapping",
    "voice swap",
    "voice shift",
    "voice change ai",
    "voice swap ai",
    "Voice changing",
    "voice change",
    "voice change real time",
    "voice changer",
    "Voice conversion",
    "ai voice conversion",
    "conversional ai voice",
    "convert voice",
    "Speech synthesis",
    "speech synthesis ai",
    "speech recognition",
    "speech to speech conversion",
    "Speech-to-text manipulation",
    "speech to text",
    "speech to text ai",
    "speech to text model",
    "speech to text generator",
    "Text-to-speech",
    "text to speech",
    "text to speech ai",
    "text to speech tool ai",
    "Attribute manipulation",
    "attribute theft",
    "manipulating elements",
    "Fake comments and reviews on social media",
    "fake comment maker",
    "fake comment ai",
    "ai for comments",
    "fake reviews",
    "fake reviews ai",
    "social media ai",
    "social media ai content",
    "ai social media post generator",
    "ai tool for social media content creation",
    "AI-generated textual content",
    "ai generated text",
    "text written by ai",
    "ai-generated content",
    "Synchronized audio-visual manipulation",
    "audio visual manipulation ai",
    "sound manipulation",
    "ai sound manipulation",
    "artificial syncronicity",
    "language manipulation",
    "Lip-sync with voice manipulation",
    "lip sync manipulation",
    "Coordinated face and voice manipulation",
    "conversation manipulation",
    "body language manipulation",
    "face voice manipulation ai",
    "tool face voice manipulation",
    "Multi-sensory deepfakes",
    "multi sensory ai",
]

# -----------------------------------------------------------------------
# CRIME TERMS (Block B) - extracted automatically from columns
# "eingabe bei reddit" + "seed term" for Main Category "Identity"
# -----------------------------------------------------------------------
CRIME_TERMS = [
    "Overcoming biometric security systems",
    "biometric verification overcome",
    "biomethric authentification",
    "facial recognition",
    "biometric access control system",
    "biometric verification overcome ai",
    "biomethric authentification ai",
    "facial recognition ai",
    "biometric access control system ai",
    "Facial recognition defeat",
    "facial recognition protection",
    "face recognition ai",
    "face recognition protection ai",
    "Voice authentication bypass",
    "voice authentication protection",
    "voice authentication ai",
    "Liveness detection circumvention",
    "liveness detection protection",
    "Multi-modal biometric system compromise",
    "multi system protection",
    "multi camera tracking",
    "biometric facial matching",
    "multi biometric protection ai",
    "false identity for financial gain",
    "assuming financial protection",
    "fake rich ai",
    "make money with ai",
    "false identity financial ai",
    "Account takeover",
    "account  takeover fraud",
    "ai takeover",
    "ai slop account",
    "Payment fraud",
    "payment fraud ai",
    "defrauding payment system",
    "ai payments",
    "crypto payment bots",
    "ai payment scam",
    "Wire transfer authorization fraud",
    "fraudulent wire transfer scam",
    "wire transfer fraud",
    "wire fraud",
    "transfer scam",
    "CEO/Executive impersonation financial transactions",
    "ceo impersonation scam",
    "boss scam",
    "ceo impersonation scam ai",
    "Fraudulent loan applications",
    "fake loan application",
    "fraudulent loan apps",
    "loan fraud ai",
    "Theft of login details",
    "login theft",
    "steal personal data",
    "personal info leak",
    "hacked and identtiy theft",
    "login theft ai",
    "steal perosnal data ai",
    "Passwords, PINs",
    "password theft",
    "pin theft",
    "passkey theft",
    "pin stolen",
    "password stolen",
    "passkey stolen",
    "Social Security Numbers",
    "social security number thread",
    "fake ssn/cc",
    "social security number ai",
    "API keys, access tokens",
    "api key theft",
    "access tokens theft",
    "access token",
    "Identity verification data",
    "ID verification",
    "biometric id verification",
    "id verification ai",
    "Falsification  official documents",
    "falsefying official document",
    "falsified document",
    "falsification private document",
    "false docuemtns",
    "fake documents",
    "fake documents ai",
    "fake receipt ai",
    "Fake IDs",
    "fake id",
    "forged signature",
    "fake id ai",
    "Fake passports",
    "false passport, fake passport ai",
    "Manipulated certificates/diplomas",
    "fake certificate",
    "fake birth certificate",
    "fake academic document",
    "fake diploma",
    "fake medical certificate",
    "fake category certificate",
    "fake transcript",
    "certificate generation",
    "fake pwd certificate",
    "fake certificate ai",
    "fake resumee",
    "Forged legal documents",
    "forged documents",
    "forging documents",
    "forged official document",
    "Manipulated government documents",
    "declassified documents",
    "declassified government documents",
    "declassified government file",
    "declassified documents ai",
    "Change  public identity/political image",
    "identity politics",
    "political identity",
    "identification with political parties",
    "image makeover in public presence",
    "identity policing",
    "public image sabotage ai",
    "Fake politician statements",
    "politicans lying",
    "fake politicians",
    "false politicians",
    "fake statements in politics",
    "fake statement politics ai",
    "Manipulated political speeches",
    "lies in poltics",
    "politician manipulation",
    "manipulation political debate",
    "politics lies ai",
    "False endorsements",
    "falsifying affiliation",
    "false affiliation",
    "false affiliation ai",
    "Fake meeting attendance",
    "falsyfing attendece records work",
    "faking meeting",
    "fake placement attendece",
    "meeting fake attendee",
    "meeting fake attendence ai",
    "Falsified policy positions",
    "false policy positions",
    "false policy positions ai",
    "Damage to professional/social reputation",
    "runiing reputation",
    "damaged reputation",
    "destroyed reputation",
    "professional reputation destroyed",
    "destroy person´s office reputation",
    "ruin professional reputation",
    "damage profession",
    "damage profession ai",
    "Professional sabotage",
    "career sabotage",
    "sabotaged job",
    "workplace sabotage",
    "employee sabotage",
    "career sabotage ai",
    "Fake scandal videos",
    "deepfake scandal video",
    "fake scandal media",
    "Business competitor harm",
    "competitor threatening",
    "business competitor hurt",
    "competitor threatening ai",
    "Romantic/social sabotage",
    "sabotating romantic relationships",
    "sabotage relationships",
    "sabotating connections",
    "sabotating love",
    "sabotating love life",
    "sabotating friendship",
    "Employment/career damage",
    "destroying career",
    "career destruction",
    "career damage ai",
    "employee damage ai",
    "Creation of completely new identities",
    "Create new identities",
    "disregard old identity",
    "get person a new identity",
    "fake identities",
    "create whole new identity",
    "abandon old identity",
    "Combination real + fake data (Frankenstein identities)",
    "create fake/synthetic data",
    "real-world synthetic data",
    "Full AI-generated personas",
    "faceless ai persona",
    "ai persona",
    "ai genrated persona",
    "Synthetic profiles on platforms",
    "synthetic users, synthetic identities social media",
    "synthetic audience",
    "Non-existent person creation fraud",
    "ai fake person to send money",
    "creating people for fraud",
    "fake persona for fraud",
    "ai fake person money",
    "Creating false personalities for deception",
    "faking personality",
    "creating fake perosnalities",
    "fake personality deception ai",
    "Dating scam profiles (Romance scams)",
    "romance scma prolies",
    "romance scam",
    "dating app scam",
    "romance scammers, dating scammers",
    "online dating scam",
    "Fake expert personas",
    "fake smart people",
    "people faking expertise",
    "pretend to be expert",
    "claim to be expert",
    "ai expert persona",
    "online fake expert",
    "Sock puppet accounts",
    "sock puppet energy",
    "sock puppet ai",
    "Catfishing identities",
    "catfish",
    "catfisch identity",
    "catfish scam",
    "catfishing",
    "catfish ai",
    "Deceptive social media profiles",
    "true identity fake social media account",
    "Non-consensual sexualized content",
    "non-consensual adult content",
    "nude videos of children",
    "non consensual sex",
    "non consensual intimate content",
    "non consensual intimate media",
    "Deepfake pornography",
    "deepfake porn",
    "deepfake pron tool",
    "deepfake ai porn",
    "ai deepfake porn",
    "deepfake/ai pornography",
    "Revenge porn",
    "temptation porn",
    "bully revenge porn",
    "Sextortion (blackmail)",
    "Sextortion",
    "blackmail",
    "black mail",
    "blackmailing",
    "blackmailed",
    "sexortion blackmailing",
    "exortion blackmail",
    "porn scma",
    "Non-consensual intimate imagery (NCII)",
    "Non-consensual intimate imagery",
    "non consensual images",
    "non consensual explicit pictures",
    "ncii",
    "non consent sex image ai",
    "Sexual identity exploitation",
    "sexual exploition",
    "sexual abuse gender identity",
    "sexual harasser identity",
    "Person is portrayed as the author of hateful statements",
    "person author hateful statement",
    "misinterpretation statement",
    "statement misinterpretation",
    "Fake racist/sexist statements",
    "Fake racist statements",
    "Fake sexist statements",
    "falsely accused",
    "racist fake",
    "fake statement ai",
    "Religious/ethnic defamation",
    "Religious defamation",
    "ethnic defamation",
    "religion hate speech",
    "religion disinformation",
    "belief defamation",
    "Character assassination",
    "Characterized assassination",
    "False controversial statements",
    "False  statements",
    "Inflammatory speech attribution",
    "inflammatory language",
    "inflammatory statements",
    "inflammatory words",
    "Imitation of authority figures",
    "misusing authority figure",
    "Email/voice phishing",
    "Email phishing",
    "Voice phishing",
    "scam mail",
    "voicemail phishing",
    "phishing calls",
    "multi channel phishing tool",
    "scam call",
    "Vishing (voice phishing)",
    "Vishing",
    "call phishing",
    "phishing alls",
    "voicemail scam",
    "Fake authority video calls",
    "fake video call scam",
    "fake video call",
    "company fake video call",
    "Government official impersonation",
    "air force impersonation scam",
    "government impersonation scam",
    "government impersonation ai",
    "Law enforcement impersonation",
    "police impersonation scam",
    "military impersonation scam",
    "fake police scam",
    "sheriff impersonation scam",
    "fbi scam impersonator",
    "police impersonation",
    "police ai",
    "Corporate executive impersonation",
    "ceo impersonation",
    "fake coperate impersonation",
]

# -----------------------------------------------------------------------
# CANDIDATE SUBREDDITS - read from your candidate discovery file
# -----------------------------------------------------------------------
# Expects a plain text file with one subreddit name per line
# (without the "r/" prefix), e.g. exported from your Excel candidate list.
CANDIDATE_SUBREDDITS_FILE = "candidate_subreddits.txt"

HEADERS = {
    "User-Agent": f"python:master-thesis-main-analysis:v1.0 (by /u/{REDDIT_USERNAME})"
}

ARCTIC_SHIFT_URL = "https://arctic-shift.photon-reddit.com/api/posts/search"

MAX_RETRIES = 4
INITIAL_BACKOFF = 10
SLEEP_BETWEEN_SUBREDDITS = 2


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_candidate_subreddits(path: str) -> list:
    try:
        with open(path, "r", encoding="utf-8") as f:
            subs = [line.strip().lstrip("r/").strip() for line in f if line.strip()]
        return subs
    except FileNotFoundError:
        print(f"ERROR: File '{path}' not found. Please place the candidate list as")
        print("a text file (one subreddit name per line) in the same folder.")
        return []


def detect_language_simple(text: str) -> str:
    """Simple language detection via langdetect (fallback: 'unknown')."""
    try:
        from langdetect import detect
        return detect(text) if text and len(text.strip()) > 10 else "unknown"
    except Exception:
        return "unknown"


def contains_any(text: str, terms: list) -> list:
    """Returns the list of terms actually found in the text
    (not just True/False), so that it can be documented WHICH terms
    triggered the hit."""
    text_lower = text.lower()
    return [t for t in terms if t.lower() in text_lower]


def fetch_subreddit_posts(subreddit: str) -> list:
    """Retrieves all historical submissions of a subreddit within the defined
    period (with pagination and retry logic)."""

    all_posts = []
    before_ts = int(END_DATE.timestamp())
    after_ts = int(START_DATE.timestamp())

    while True:
        params = {
            "subreddit": subreddit,
            "after": after_ts,
            "before": before_ts,
            "limit": 100,
            "sort": "desc",
        }

        backoff = INITIAL_BACKOFF
        response = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = requests.get(ARCTIC_SHIFT_URL, params=params,
                                         headers=HEADERS, timeout=30)
                if response.status_code == 429:
                    print(f"    Rate limit hit, waiting {backoff}s...")
                    time.sleep(backoff)
                    backoff *= 2
                    continue
                response.raise_for_status()
                break
            except requests.exceptions.RequestException as e:
                print(f"    Error (attempt {attempt}/{MAX_RETRIES}): {e}")
                time.sleep(backoff)
                backoff *= 2
                response = None
        else:
            print(f"    Aborted after {MAX_RETRIES} attempts.")
            break

        if response is None:
            break

        data = response.json().get("data", [])
        if not data:
            break

        all_posts.extend(data)

        oldest_ts = min(p.get("created_utc", after_ts) for p in data)
        if oldest_ts <= after_ts or len(data) < 100:
            break
        before_ts = oldest_ts

        time.sleep(0.5)

    return all_posts


# =============================================================================
# MAIN WORKFLOW
# =============================================================================

def main():
    if REDDIT_USERNAME == "your_reddit_username":
        print("WARNING: Please set REDDIT_USERNAME at the top of the script first!\n")
        return

    print(f"Loaded search terms: {len(MODALITY_TERMS)} modality terms, "
          f"{len(CRIME_TERMS)} crime terms.\n")

    candidates = load_candidate_subreddits(CANDIDATE_SUBREDDITS_FILE)
    if not candidates:
        return

    print(f"Starting data collection for {len(candidates)} candidate subreddits...")
    print(f"Period: {START_DATE.date()} to {END_DATE.date()}\n")

    all_rows = []

    for i, subreddit in enumerate(candidates, start=1):
        print(f"[{i}/{len(candidates)}] r/{subreddit}")
        posts = fetch_subreddit_posts(subreddit)
        print(f"  {len(posts)} submissions retrieved in total.")

        hits_in_sub = 0
        for post in posts:
            title = post.get("title", "") or ""
            selftext = post.get("selftext", "") or ""
            combined_text = f"{title} {selftext}"

            is_removed = selftext.strip() in ("[deleted]", "[removed]", "")

            matched_modality = contains_any(combined_text, MODALITY_TERMS)
            matched_crime = contains_any(combined_text, CRIME_TERMS)

            retrieval_hit = bool(matched_modality) and bool(matched_crime)

            if not retrieval_hit:
                continue

            language = detect_language_simple(combined_text)
            if language not in ALLOWED_LANGUAGES and language != "unknown":
                continue

            hits_in_sub += 1
            all_rows.append({
                "post_id": post.get("id"),
                "subreddit": subreddit,
                "created_utc": post.get("created_utc"),
                "created_date": datetime.fromtimestamp(
                    post.get("created_utc", 0), tz=timezone.utc
                ).strftime("%Y-%m-%d") if post.get("created_utc") else None,
                "title": title,
                "selftext": selftext,
                "permalink": post.get("permalink"),
                "score": post.get("score"),
                "num_comments": post.get("num_comments"),
                "language": language,
                "matched_modality_terms": "; ".join(matched_modality),
                "matched_crime_terms": "; ".join(matched_crime),
                "retrieval_hit": 1,
                "is_removed_or_empty": is_removed,
                "content_relevant": "",
                "exclusion_reason": "",
                "modality_code": "",
                "crime_code": "",
            })

        print(f"  -> {hits_in_sub} Candidate Hits (Modality AND Crime).")
        time.sleep(SLEEP_BETWEEN_SUBREDDITS)

    if not all_rows:
        print("\nNo candidate hits found. Please check search terms/period.")
        return

    df = pd.DataFrame(all_rows)

    before_dedup = len(df)
    df = df.drop_duplicates(subset=["post_id"])
    print(f"\nDeduplication: {before_dedup} -> {len(df)} unique candidate posts.")

    df.to_csv("candidate_posts_full.csv", index=False)
    print("Saved -> candidate_posts_full.csv")

    summary = (
        df.groupby("subreddit")
        .agg(candidate_hits=("post_id", "count"))
        .reset_index()
        .sort_values("candidate_hits", ascending=False)
    )
    summary.to_csv("discourse_volume_per_subreddit.csv", index=False)
    print("Saved -> discourse_volume_per_subreddit.csv")
    print(f"\n{len(summary)} of {len(candidates)} subreddits had at least one candidate hit.")
    print("\nTop 15 by number of candidate hits (descriptive only, not a selection criterion):")
    print(summary.head(15).to_string(index=False))

    print("\nNEXT STEP: Use candidate_posts_full.csv for the manual")
    print("content validation (fill in the columns 'content_relevant', 'exclusion_reason')")
    print("to separate retrieval hits from posts that are actually relevant in")
    print("terms of content.")


if __name__ == "__main__":
    main()